# 05 – DistilBERT Data Preparation

**Purpose:** Build and verify the data pipelines for future fine-tuning of a `DistilBERT` model.

This notebook covers:
1. Loading the preprocessed training and test splits.
2. Initializing `TransformerTicketDataset` with automatic Hugging Face tokenization.
3. Demonstrating local caching of pre-tokenized features.
4. Constructing a PyTorch `DataLoader` with our custom `DynamicPaddingCollator`.
5. Verifying that batch shapes fluctuate dynamically depending on sequence lengths.

## 0. Setup and Imports

In [1]:
import sys
from pathlib import Path

from torch.utils.data import DataLoader

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT / "SupportAI").exists():
    REPO_ROOT = REPO_ROOT / "SupportAI"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: C:\Users\gunav\Downloads\SupportAI


## 1. Load Preprocessed Data Splits

We load train and test splits using our data ingestion module.

In [2]:
from src.data.dataset import load_and_preprocess_dataset

splits = load_and_preprocess_dataset()
train_df = splits["train"]
test_df = splits["test"]

print(f"Train records: {len(train_df)}")
print(f"Test records:  {len(test_df)}")

[07/13/26 20:30:30] INFO     All dataset splits cached locally under:                                              
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1. Loading...

                    INFO     Loading 'train' split from local cache:                                               
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\train.parquet

                    INFO     Loading 'val' split from local cache:                                                 
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\val.parquet

                    INFO     Loading 'test' split from local cache:                                                
                             C:\Users\gunav\Downloads\SupportAI\data\customer_support_tickets_v1\test.parquet

Train records: 10458
Test records:  1302


## 2. Initialize TransformerTicketDataset and Caching

We construct the dataset and cache the features locally to avoid redundant Hugging Face tokenization.

In [3]:
from src.models.transformer.dataset import TransformerTicketDataset

cache_path = REPO_ROOT / "data" / "transformer_cache" / "train_tokenized.pt"

train_dataset = TransformerTicketDataset(
    texts=train_df["text"].tolist(),
    labels=train_df["label"].tolist(),
    model_name="distilbert-base-uncased",
    max_length=128,
    cache_path=cache_path,
    use_cache=True,
)

print(f"Dataset initialized. Length: {len(train_dataset)}")
print("Sample items retrieved:")
sample = train_dataset[0]
print(f"  - input_ids shape: {sample['input_ids'].shape}")
print(f"  - label:           {sample['label']}")

[07/13/26 20:30:35] INFO     Loading faiss with AVX2 support.

                    INFO     Could not load library with AVX2 support due to:                                      
                             ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")

                    INFO     Loading faiss.

                    INFO     Successfully loaded faiss.

[07/13/26 20:30:36] INFO     Loading tokenizer: distilbert-base-uncased

[07/13/26 20:30:38] INFO     Loading pre-tokenized cache from:                                                     
                             C:\Users\gunav\Downloads\SupportAI\data\transformer_cache\train_tokenized.pt

Dataset initialized. Length: 10458
Sample items retrieved:
  - input_ids shape: torch.Size([11])
  - label:           38


## 3. Verify Batch Creation & Dynamic Padding Collator

We create a PyTorch DataLoader and verify that dynamic padding keeps sequence dimensions compact.

In [4]:
from src.models.transformer.collator import DynamicPaddingCollator

# Custom collator using the tokenizer's padding token ID
collator = DynamicPaddingCollator(pad_token_id=train_dataset.tokenizer.pad_token_id)

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collator,
)

print("Iterating first 5 batches and printing padded shapes:")
for batch_idx, batch in enumerate(loader):
    if batch_idx >= 5:
        break
    print(f"Batch {batch_idx + 1}:")
    print(f"  - input_ids shape:      {batch['input_ids'].shape}")
    print(f"  - attention_mask shape: {batch['attention_mask'].shape}")
    print(f"  - labels shape:         {batch['labels'].shape}")

Iterating first 5 batches and printing padded shapes:
Batch 1:
  - input_ids shape:      torch.Size([8, 28])
  - attention_mask shape: torch.Size([8, 28])
  - labels shape:         torch.Size([8])
Batch 2:
  - input_ids shape:      torch.Size([8, 18])
  - attention_mask shape: torch.Size([8, 18])
  - labels shape:         torch.Size([8])
Batch 3:
  - input_ids shape:      torch.Size([8, 37])
  - attention_mask shape: torch.Size([8, 37])
  - labels shape:         torch.Size([8])
Batch 4:
  - input_ids shape:      torch.Size([8, 32])
  - attention_mask shape: torch.Size([8, 32])
  - labels shape:         torch.Size([8])
Batch 5:
  - input_ids shape:      torch.Size([8, 66])
  - attention_mask shape: torch.Size([8, 66])
  - labels shape:         torch.Size([8])


In [5]:
# Export Phase Manifest
from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml
from src.utils.config import load_config

config = load_config()
manifest = {
    "phase": "05_Distil_Data",
    "max_length": config.get("max_length", 128),
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_05_distil_data.yaml")
print("YAML manifest saved successfully:")
print(manifest)


[07/13/26 20:30:40] INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_05_distil_data.yaml

YAML manifest saved successfully:
{'phase': '05_Distil_Data', 'max_length': 128, 'git_commit': 'ef9a0498221c5c43373fcf9e951987614174868f'}
